# 03a · A sarcasm direction in the residual stream

Phase 2 showed that the mechanism is **distributed**: no small circuit answers like the model. Here we ask what the
full model **represents**, independently of any circuit: is there a direction in the residual stream that encodes
"this is sarcastic", from which layer, and is it causal?

1. **Probes per layer** (last token): a linear classifier S vs (P, N), trained on discovery prompts and tested on
   held-out prompts with **new phrases** and a **new sentence frame**. Same for two controls: phrase polarity
   (positive vs negative phrase) and situation valence (pleasant vs unpleasant).
2. **More than sentiment?** A 2 × 2 design (adding pleasant situation + negative phrase): is there an
   *interaction* (incongruity) signal beyond the two sentiment features, and is it the same on new phrases / frames?
3. **Steering (causal test)**: add α · (mean S − mean P/N) to the residual stream at the last token at one layer:
   do non-sarcastic prompts become "Yes" (α > 0) and sarcastic ones "No" (α < 0)? Control: a random direction of
   the same length.
4. **Who writes the direction**: the projection of every head's and MLP's output on the direction, S vs P/N, and
   whether these components are in the ACDC circuits.
5. **Your own sentences**: where they fall on the direction (0 = like an average non-sarcastic prompt,
   1 = like an average sarcastic prompt).

**Runtime:** T4 is enough; about 20–30 minutes (the residual streams of all prompts are saved on Drive, ~0.6 GB).

## Setup

In [ ]:
# Get (or update) the code from GitHub
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q -U accelerate bitsandbytes

In [ ]:
import os, json, time, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
from src.model import load_model, model_info, DEV_MODEL, MAIN_MODEL
from src.prompts import load_dataset, encode, chat_text, data_tag
from src.direction import collect_resid, steer_d, component_writes, logreg_predict
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
MODEL = MAIN_MODEL          # DEV_MODEL (3B) for quick tests
DATA = "v2"                 # dataset: "v1" (pilot) or "v2"
STEER_LAYERS = [6, 10, 12, 14, 16, 18, 20, 22, 24]
ALPHAS = [-2, -1, -0.5, 0, 0.5, 1, 2]       # multiples of (mean S - mean P/N)
N_STEER = 40                # held-out triples per held-out group for steering
N_WRITE = 100               # discovery triples for "who writes the direction"
# ------------------------------------------
TAG_MODEL = MODEL.split("/")[-1]
TAG = TAG_MODEL + data_tag(DATA)
SUB = f"03a_direction/{TAG}"
mount_drive()
print("results go to:", results_dir(SUB))
set_seed(0)

In [ ]:
t0 = time.time()
model, tok = load_model(MODEL)
INFO = model_info(model)
L, H = INFO["n_layers"], INFO["n_heads"]
print(MODEL, INFO, f"loaded in {(time.time() - t0) / 60:.1f} min")

df = load_dataset(DATA).sort_values(["split", "triple_id", "condition"]).reset_index(drop=True)
df["group"] = [s if s == "discovery" else f"heldout ({t})" for s, t in zip(df.split, df.heldout_type)]
GROUPS = [g for g in ["discovery", "heldout (phrase)", "heldout (frame)"] if g in set(df.group)]
IDS = [encode(tok, r).ids for r in df.itertuples()]
TRAIN = (df.group == "discovery").values
LABELS = {
    "sarcasm (S vs P, N)": (df.condition == "S").values,
    "phrase polarity (positive)": (df.frame_type == "positive").values,
    "situation valence (pleasant)": (df.situation_type == "pleasant").values,
}
print({g: int((df.group == g).sum()) for g in GROUPS}, "prompts")

## Residual stream at the last token, all layers

In [ ]:
t0 = time.time()
X, D_FULL = cached("resid_last_token", lambda: collect_resid(model, tok, IDS), sub=SUB)
print("X", X.shape, X.dtype, f"| {(time.time() - t0) / 60:.1f} min")
df["d full"] = D_FULL

## 1 · Probes per layer

Logistic regression (L2, fitted on the GPU) on the standardised residual stream (trained on discovery), balanced accuracy on each group
(0.5 = chance). Held-out groups are never seen in training.

In [ ]:
def bal_acc(pred, y):
    return 0.5 * (pred[y].mean() + (~pred[~y]).mean())

def layer_feats(l):
    Z = X[:, l].astype(np.float32)
    mu, sd = Z[TRAIN].mean(0), Z[TRAIN].std(0) + 1e-6
    return (Z - mu) / sd

def run_probes():
    out = {}
    for name, y in LABELS.items():
        acc = np.zeros((L + 1, len(GROUPS)))
        for l in range(L + 1):
            Z = layer_feats(l)
            pred = logreg_predict(Z[TRAIN], y[TRAIN], Z)
            for j, g in enumerate(GROUPS):
                m = (df.group == g).values
                acc[l, j] = bal_acc(pred[m], y[m])
        out[name] = acc
        print(f"  {name}: done")
    return out

t0 = time.time()
PROBE = cached("probes", run_probes, sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, (name, acc) in zip(axes, PROBE.items()):
    for j, g in enumerate(GROUPS):
        ax.plot(acc[:, j], marker="o", ms=3, label=g)
    ax.axhline(0.5, c="gray", lw=0.6); ax.set_title(name); ax.set_xlabel("layer (input of)")
axes[0].set_ylabel("balanced accuracy"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "probes_per_layer.png"), dpi=150); plt.show()

def mean_dir(l, y):
    Z = X[:, l].astype(np.float32)
    return Z[TRAIN & y].mean(0) - Z[TRAIN & ~y].mean(0)

sarc = PROBE["sarcasm (S vs P, N)"]
held = [j for j, g in enumerate(GROUPS) if g != "discovery"]
L_STAR = int(np.argmax(sarc[:L, held].mean(1)))      # an input of a layer (0 .. L-1), so we can steer there
print(f"best layer for sarcasm on held-out prompts: {L_STAR} "
      f"(balanced accuracy {dict(zip(GROUPS, sarc[L_STAR].round(3)))})")

## 2 · More than sentiment? (2 × 2 design)

With only P, S, N every difference between class means lies in the plane of the phrase and situation directions
(algebra, not data), so "remove the sentiment directions" cannot work with three conditions. We add a **fourth**
condition, **Q = pleasant situation + negative phrase** (the same sentences as P with N's phrase), and get a 2 × 2
design (situation pleasant / unpleasant × phrase positive / negative; sarcastic only in S). Per layer:
* **main effects**: phrase φ = mean(P, S) − mean(N, Q), situation σ = mean(P, Q) − mean(S, N);
* **interaction** ι = (S − N) − (P − Q): how much the effect of the phrase depends on the situation. If the model only
  added up the two sentiments, ι = 0; an *incongruity* signal makes it large;
* is ι **the same** on new phrases and the new frame (cosine between ι from discovery and from held-out)?
* **probe for S without the main effects**: remove φ and σ from the residual stream and classify S vs (P, N, Q).

In [ ]:
# the fourth condition Q: P's sentence with N's phrase
q_rows = []
for tid, g in df.groupby("triple_id"):
    r = g.set_index("condition")
    pos, neg = r.loc["P", "frame"], r.loc["N", "frame"]
    text = r.loc["P", "text"][: -len(pos)] + neg
    q_rows.append(dict(triple_id=tid, condition="Q", text=text, group=r.loc["P", "group"]))
QD = pd.DataFrame(q_rows)
template = df.prompt.iloc[0].replace(df.text.iloc[0], "{}")
Q_IDS = [tok(chat_text(tok, template.format(t)), add_special_tokens=False)["input_ids"] for t in QD.text]
XQ, DQ = cached("resid_last_token_Q", lambda: collect_resid(model, tok, Q_IDS), sub=SUB)
QD["d full"] = DQ
print("examples:", QD.text.iloc[0], "|", QD.text.iloc[500])
print("full model 'Yes' rate on Q (pleasant + negative phrase):",
      {g: round(float((QD[QD.group == g]["d full"] > 0).mean()), 3) for g in GROUPS})

# 4-condition arrays
X4 = np.concatenate([X, XQ])
C4 = np.concatenate([df.condition.values, QD.condition.values])
G4 = np.concatenate([df.group.values, QD.group.values])
TR4 = G4 == "discovery"

def effects(l, mask):
    Z = X4[:, l].astype(np.float32)
    m = {c: Z[mask & (C4 == c)].mean(0) for c in "PSNQ"}
    phi = (m["P"] + m["S"]) / 2 - (m["N"] + m["Q"]) / 2
    sig = (m["P"] + m["Q"]) / 2 - (m["S"] + m["N"]) / 2
    iota = (m["S"] - m["N"]) - (m["P"] - m["Q"])
    return phi, sig, iota

def interaction_test():
    out = dict(ratio=np.zeros(L + 1), cos_held=np.zeros((L + 1, len(GROUPS) - 1)), probe=np.zeros((L + 1, len(GROUPS))))
    y = C4 == "S"
    for l in range(L + 1):
        phi, sig, iota = effects(l, TR4)
        out["ratio"][l] = np.linalg.norm(iota) / (np.linalg.norm(phi) + np.linalg.norm(sig))
        for j, g in enumerate(GROUPS[1:]):
            _, _, io_h = effects(l, G4 == g)
            out["cos_held"][l, j] = iota @ io_h / np.linalg.norm(iota) / np.linalg.norm(io_h)
        Z = X4[:, l].astype(np.float32)
        Qb, _ = np.linalg.qr(np.stack([phi, sig], 1))
        Zr = Z - (Z @ Qb) @ Qb.T
        mu, sd = Zr[TR4].mean(0), Zr[TR4].std(0) + 1e-6
        Fz = (Zr - mu) / sd
        pred = logreg_predict(Fz[TR4], y[TR4], Fz)
        for j, g in enumerate(GROUPS):
            m = G4 == g
            out["probe"][l, j] = bal_acc(pred[m], y[m])
    return out

IT = cached("interaction_test", interaction_test, sub=SUB)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(IT["ratio"], marker="o", ms=3); axes[0].set_title("|interaction| / (|phrase| + |situation|)")
for j, g in enumerate(GROUPS[1:]):
    axes[1].plot(IT["cos_held"][:, j], marker="o", ms=3, label=g)
axes[1].set_title("same interaction on held-out? cos(ι disc, ι held-out)"); axes[1].axhline(0, c="gray", lw=0.6); axes[1].legend(fontsize=8)
for j, g in enumerate(GROUPS):
    axes[2].plot(IT["probe"][:, j], marker="o", ms=3, label=g)
axes[2].axhline(0.5, c="gray", lw=0.6); axes[2].set_title("S vs (P, N, Q) without phrase + situation directions"); axes[2].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("layer")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "more_than_sentiment.png"), dpi=150); plt.show()
print(f"layer {L_STAR}: interaction ratio {IT['ratio'][L_STAR]:.2f}, cos with held-out {IT['cos_held'][L_STAR].round(2)}, "
      f"probe without main effects {dict(zip(GROUPS, IT['probe'][L_STAR].round(3)))}")

## 3 · Steering (causal test)

At the input of a layer, add α · v to the residual stream at the last token, v = mean(S) − mean(P, N) at that layer
(discovery). α = +1 moves an average non-sarcastic prompt to the average sarcastic one. Held-out prompts only.

In [ ]:
rng = np.random.default_rng(0)
steer_idx = []
for g in GROUPS[1:]:
    tids = rng.choice(df[df.group == g].triple_id.unique(), min(N_STEER, df[df.group == g].triple_id.nunique()), replace=False)
    steer_idx += list(df.index[df.triple_id.isin(tids)])
SD = df.loc[steer_idx]
print(f"{len(SD)} held-out prompts for steering")

V = {l: mean_dir(l, LABELS["sarcasm (S vs P, N)"]) for l in range(L + 1)}

def run_steering():
    out = {}
    for l in STEER_LAYERS:
        out[l] = steer_d(model, tok, [IDS[i] for i in steer_idx], l, V[l], ALPHAS)
        print(f"  layer {l} done")
    r = np.random.default_rng(1).standard_normal(V[L_STAR].shape).astype(np.float32)
    r *= np.linalg.norm(V[L_STAR]) / np.linalg.norm(r)
    out["random"] = steer_d(model, tok, [IDS[i] for i in steer_idx], L_STAR, r, ALPHAS)
    return out

t0 = time.time()
STEER = cached(f"steering_{'_'.join(map(str, STEER_LAYERS))}_n{N_STEER}_Lstar{L_STAR}", run_steering, sub=SUB)
print(f"{(time.time() - t0) / 60:.1f} min")

cond = SD.condition.values
a0 = ALPHAS.index(0)
rows = []
for key, dd in STEER.items():
    for k, al in enumerate(ALPHAS):
        rows.append(dict(layer=key, alpha=al,
                         **{f"Yes rate {c}": float((dd[k][cond == c] > 0).mean()) for c in "PSN"},
                         **{f"mean d {c}": float(dd[k][cond == c].mean()) for c in "PSN"}))
ST = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, c in zip(axes, "PSN"):
    for key in STEER:
        s = ST[ST.layer == key]
        ax.plot(s.alpha, s[f"Yes rate {c}"], marker="o", ms=3, label=f"layer {key}" if key != "random" else f"random (layer {L_STAR})",
                lw=2.5 if key == L_STAR else 1, ls=":" if key == "random" else "-", color="black" if key == "random" else None)
    ax.set_title(f"{c} prompts"); ax.set_xlabel("α (× mean S − mean P,N)"); ax.axvline(0, c="gray", lw=0.5)
axes[0].set_ylabel("share of 'Yes'"); axes[-1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "steering.png"), dpi=150); plt.show()

summary = ST[ST.alpha.isin([-1, 0, 1])].pivot(index="layer", columns="alpha", values=["Yes rate P", "Yes rate S", "Yes rate N"])
display(summary.round(2))

## 4 · Who writes the direction

At the best layer `L_STAR`, the projection of every head's and MLP's output (last token) on the unit sarcasm
direction; score = mean over S − mean over P, N. Sanity: embedding + all heads + all MLPs = the residual stream.
Components are marked if they have edges in the ACDC circuits (v3b τ = 0) found on v2 and on v1.

In [ ]:
rng = np.random.default_rng(2)
tids = rng.choice(df[df.group == "discovery"].triple_id.unique(), N_WRITE, replace=False)
w_idx = list(df.index[df.triple_id.isin(tids)])
u = V[L_STAR] / np.linalg.norm(V[L_STAR])
HW, MW = cached(f"writes_L{L_STAR}_n{N_WRITE}", lambda: component_writes(model, tok, [IDS[i] for i in w_idx], u, L_STAR), sub=SUB)

# sanity: the parts add up to the residual stream projection
total = X[w_idx, 0].astype(np.float32) @ u + HW.sum((1, 2)) + MW.sum(1)
direct = X[w_idx, L_STAR].astype(np.float32) @ u
print(f"sanity: corr(sum of parts, residual) = {np.corrcoef(total, direct)[0, 1]:.4f}, max |diff| = {np.abs(total - direct).max():.3f}")

cw = df.loc[w_idx].condition.values
score_h = HW[cw == "S"].mean(0) - HW[cw != "S"].mean(0)        # [L_STAR, H]
score_m = MW[cw == "S"].mean(0) - MW[cw != "S"].mean(0)        # [L_STAR]
gap = direct[cw == "S"].mean() - direct[cw != "S"].mean()
print(f"S - P/N gap along the direction at layer {L_STAR}: {gap:.2f} (embedding part {(X[w_idx, 0].astype(np.float32) @ u)[cw == 'S'].mean() - (X[w_idx, 0].astype(np.float32) @ u)[cw != 'S'].mean():.2f})")

def in_circuit(path, l, h=None):
    p = os.path.join(results_dir(""), path)
    if not os.path.exists(p):
        return None
    m = np.load(p)
    row = 1 + l * H + h if h is not None else 1 + L * H + l
    return bool(m[row].any())
C2 = f"02_acdc/{TAG}/v3b/acdc_mask_tau0.0.npy"
C1 = f"02_acdc/{TAG_MODEL}/v3b/acdc_mask_tau0.0.npy"
rows = [dict(component=f"a{l}.h{h}", layer=l, write=score_h[l, h]) for l in range(L_STAR) for h in range(H)]
rows += [dict(component=f"m{l}", layer=l, write=score_m[l]) for l in range(L_STAR)]
W = pd.DataFrame(rows)
W["share of gap"] = W.write / gap
W["in ACDC v2"] = [in_circuit(C2, r.layer, int(r.component.split(".h")[1]) if ".h" in r.component else None) for r in W.itertuples()]
W["in ACDC v1"] = [in_circuit(C1, r.layer, int(r.component.split(".h")[1]) if ".h" in r.component else None) for r in W.itertuples()]
W = W.reindex(W.write.abs().sort_values(ascending=False).index).reset_index(drop=True)
display(W.head(25).round(3))
top = W.head(25)
print(f"top-25 writers: {int(top['in ACDC v2'].sum())} in the ACDC v2 circuit, {int(top['in ACDC v1'].sum())} in the v1 circuit")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [3, 1]})
lim = np.abs(score_h).max()
im = axes[0].imshow(score_h.T, aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, origin="lower")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("head"); axes[0].set_title(f"heads: write along the sarcasm direction (S − P/N), read at layer {L_STAR}")
plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[1].barh(range(L_STAR), score_m, color=["#C62828" if s > 0 else "#1565C0" for s in score_m])
axes[1].set_xlabel("MLP write"); axes[1].set_ylabel("layer"); axes[1].set_title("MLPs")
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "who_writes.png"), dpi=150); plt.show()

## 5 · Your own sentences

Score = position on the sarcasm direction at layer `L_STAR`: 0 = like an average non-sarcastic discovery prompt,
1 = like an average sarcastic one. Next to it the full model's answer. The histogram shows the scores of the
held-out prompts for reference.

In [ ]:
SENTENCES = [
    "My house burned down. This is the happiest day of my life.",
    "My house burned down. This is the worst day of my life.",
    "I won a free trip to Paris. This is the happiest day of my life.",
    "I got to work on my weekend. What a fantastic way to relax.",
    "I got to sleep in all weekend. What a fantastic way to relax.",
    "My dog died today. This is the best news I had all day.",
    "Oh great, another Monday.",
    "Thanks a lot for nothing.",
]
template = df.prompt.iloc[0].replace(df.text.iloc[0], "{}")
ids_own = [tok(chat_text(tok, template.format(s)), add_special_tokens=False)["input_ids"] for s in SENTENCES]
Xo, Do = collect_resid(model, tok, ids_own, progress=False)

proj = lambda Z: Z.astype(np.float32) @ u
p_all = proj(X[:, L_STAR])
lo, hi = p_all[TRAIN & (df.condition != "S").values].mean(), p_all[TRAIN & (df.condition == "S").values].mean()
score = lambda p: (p - lo) / (hi - lo)
for s, p, d in zip(SENTENCES, proj(Xo[:, L_STAR]), Do):
    print(f"score {score(p):+5.2f}  |  full model d = {d:+6.2f} -> {'Yes' if d > 0 else 'No '}  |  {s}")

fig, ax = plt.subplots(figsize=(9, 3))
for c, col in [("P", "#1565C0"), ("S", "#C62828"), ("N", "#2E7D32")]:
    m = (~TRAIN) & (df.condition == c).values
    ax.hist(score(p_all[m]), bins=40, alpha=0.5, color=col, label=f"held-out {c}")
for p in proj(Xo[:, L_STAR]):
    ax.axvline(score(p), c="black", lw=0.8)
ax.set_xlabel(f"score on the sarcasm direction (layer {L_STAR}); black lines = your sentences"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(results_dir(SUB), "own_sentences.png"), dpi=150); plt.show()

## Save

In [ ]:
d = results_dir(SUB)
for name, acc in PROBE.items():
    pd.DataFrame(acc, columns=GROUPS).to_csv(os.path.join(d, f"probe_{name.split(' ')[0]}.csv"), index_label="layer")
ST.to_csv(os.path.join(d, "steering.csv"), index=False)
W.to_csv(os.path.join(d, "who_writes.csv"), index=False)
np.save(os.path.join(d, f"sarcasm_direction_L{L_STAR}.npy"), V[L_STAR])
json.dump(dict(model=MODEL, data=DATA, L_STAR=L_STAR,
               probe_sarcasm_at_L_STAR=dict(zip(GROUPS, PROBE["sarcasm (S vs P, N)"][L_STAR].round(4).tolist())),
               interaction_ratio_at_L_STAR=float(IT["ratio"][L_STAR]),
               interaction_cos_heldout_at_L_STAR=IT["cos_held"][L_STAR].round(4).tolist(),
               probe_without_main_effects_at_L_STAR=dict(zip(GROUPS, IT["probe"][L_STAR].round(4).tolist()))),
          open(os.path.join(d, "summary.json"), "w"), indent=1)
print("saved to", d)